# 🩺 DiabetesClassifier — Predicting Diabetes Diagnosis

**Dataset:** Synthetic healthcare dataset — 100,000 records, 31 features  
**Target:** `diagnosed_diabetes` (0 = No Diabetes, 1 = Diagnosed)  
**Models:** Random Forest · Logistic Regression  

---
### Pipeline
1. Imports & Setup
2. Load Data & EDA
3. Preprocessing
4. Random Forest (GridSearchCV)
5. Logistic Regression (GridSearchCV)
6. Evaluation & Comparison
7. Feature Importance & Coefficients
8. Save Outputs

## 1 — Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve, f1_score
)

os.makedirs('outputs/figures', exist_ok=True)
os.makedirs('outputs/models',  exist_ok=True)

print('Setup complete.')

## 2 — Load Data & EDA

In [ ]:
df = pd.read_csv('data/diabetes_dataset.csv')

print(f'Shape: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Duplicate rows : {df.duplicated().sum()}')
df.head()

In [ ]:
# --- Descriptive statistics ---
df.describe(include='all')

In [ ]:
# --- Target class distribution ---
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['diagnosed_diabetes'].value_counts()
ax.bar(['No Diabetes (0)', 'Diabetes (1)'], counts.values,
       color=['#5cb85c', '#d9534f'], edgecolor='white', width=0.5)
for i, v in enumerate(counts.values):
    ax.text(i, v + 500, f'{v:,}\n({v/len(df)*100:.1f}%)',
            ha='center', fontsize=11)
ax.set_title('Target Class Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('outputs/figures/class_distribution.png', dpi=150)
plt.show()
print(f'Class balance: {counts[0]:,} negative  |  {counts[1]:,} positive')

In [ ]:
# --- Correlation heatmap (numeric features only) ---
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numerical_cols].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm',
            linewidths=0.4, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# --- Top correlations with target ---
target_corr = corr['diagnosed_diabetes'].drop('diagnosed_diabetes') \
              .abs().sort_values(ascending=False)
print('Top 10 features correlated with diagnosed_diabetes:')
print(target_corr.head(10).to_string())

In [ ]:
# --- Distribution of key clinical features by diagnosis ---
key_features = ['hba1c', 'glucose_fasting', 'bmi',
                'diabetes_risk_score', 'insulin_level']

fig, axes = plt.subplots(1, len(key_features), figsize=(20, 4))
for ax, feat in zip(axes, key_features):
    for label, colour in [(0, '#5cb85c'), (1, '#d9534f')]:
        subset = df[df['diagnosed_diabetes'] == label][feat]
        ax.hist(subset, bins=40, alpha=0.6, color=colour,
                label='No Diabetes' if label == 0 else 'Diabetes')
    ax.set_title(feat.replace('_', '\n'), fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Key Feature Distributions by Diagnosis', y=1.02,
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/feature_distributions.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 3 — Preprocessing

In [ ]:
# --- Encode categorical columns ---
# LabelEncoder maps each unique string to an integer.
# One-hot encoding would create too many dummy columns
# for tree-based models; label encoding is sufficient here.
categorical_cols = [
    'gender', 'ethnicity', 'education_level',
    'income_level', 'employment_status', 'smoking_status'
]
le = LabelEncoder()
df_enc = df.copy()
for col in categorical_cols:
    df_enc[col] = le.fit_transform(df_enc[col])

# --- Drop target-leakage column ---
# diabetes_stage directly encodes the target (e.g. 'Type 2', 'No Diabetes')
# and must be removed to prevent data leakage.
df_enc.drop(columns=['diabetes_stage'], inplace=True)

# --- Features and target ---
X = df_enc.drop(columns=['diagnosed_diabetes'])
y = df_enc['diagnosed_diabetes']

print(f'Feature matrix : {X.shape}')
print(f'Target series  : {y.shape}')
print(f'Features       : {list(X.columns)}')

In [ ]:
# --- Train / test split (stratified 80/20) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Scale features ---
# StandardScaler is critical for Logistic Regression (gradient-based)
# and does not harm Random Forest (tree splits are scale-invariant).
# Fit ONLY on training data to prevent data leakage.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# --- Class weights ---
# Computed from training labels only.
# Balances the loss function so the minority class is not ignored.
cw_values = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weights = {0: cw_values[0], 1: cw_values[1]}

print(f'Training set   : {X_train_sc.shape[0]:,} rows')
print(f'Test set       : {X_test_sc.shape[0]:,} rows')
print(f'Class weights  : {class_weights}')

## 4 — Random Forest

In [ ]:
rf = RandomForestClassifier(
    class_weight=class_weights,
    random_state=42,
    n_jobs=-1
)

param_grid_rf = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 10, 20],
    'min_samples_split': [2, 5, 10],
}

gs_rf = GridSearchCV(
    rf, param_grid_rf,
    cv=5, scoring='roc_auc',
    n_jobs=-1, verbose=1
)
gs_rf.fit(X_train_sc, y_train)

best_rf = gs_rf.best_estimator_
print(f'Best RF params : {gs_rf.best_params_}')
print(f'CV ROC-AUC     : {gs_rf.best_score_:.4f}')

## 5 — Logistic Regression

In [ ]:
lr = LogisticRegression(
    class_weight=class_weights,
    random_state=42,
    max_iter=1000
)

param_grid_lr = {
    'C':      [0.001, 0.01, 0.1, 1.0, 10.0],
    'solver': ['lbfgs', 'liblinear'],
}

gs_lr = GridSearchCV(
    lr, param_grid_lr,
    cv=5, scoring='roc_auc',
    n_jobs=-1, verbose=1
)
gs_lr.fit(X_train_sc, y_train)

best_lr = gs_lr.best_estimator_
print(f'Best LR params : {gs_lr.best_params_}')
print(f'CV ROC-AUC     : {gs_lr.best_score_:.4f}')

## 6 — Evaluation & Comparison

In [ ]:
results = {}

for name, model in [('Random Forest', best_rf), ('Logistic Regression', best_lr)]:
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    auc  = roc_auc_score(y_test, y_prob)
    results[name] = dict(accuracy=acc, f1=f1, roc_auc=auc,
                         y_pred=y_pred, y_prob=y_prob)
    print(f'\n{"="*50}')
    print(f'  {name}')
    print(f'{"="*50}')
    print(classification_report(y_test, y_pred,
                                 target_names=['No Diabetes', 'Diabetes']))
    print(f'ROC-AUC : {auc:.4f}')

In [ ]:
# --- Confusion matrices ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Diabetes', 'Diabetes'],
                yticklabels=['No Diabetes', 'Diabetes'])
    ax.set_title(f'{name}\nConfusion Matrix', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('outputs/figures/confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# --- ROC curve overlay ---
fig, ax = plt.subplots(figsize=(8, 6))
colours = {'Random Forest': 'steelblue', 'Logistic Regression': 'tomato'}

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, lw=2, color=colours[name],
            label=f"{name}  (AUC = {res['roc_auc']:.3f})")

ax.plot([0,1],[0,1], 'k--', lw=1, label='Random baseline (AUC = 0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('outputs/figures/roc_comparison.png', dpi=150)
plt.show()

In [ ]:
# --- Summary metrics table ---
summary = pd.DataFrame({
    name: {
        'Accuracy':  f"{res['accuracy']:.4f}",
        'F1 (wtd)':  f"{res['f1']:.4f}",
        'ROC-AUC':   f"{res['roc_auc']:.4f}",
    }
    for name, res in results.items()
})
print(summary.to_string())
summary.to_csv('outputs/results_summary.csv')

## 7 — Feature Importance & Coefficients

In [ ]:
# --- Random Forest: Gini feature importance ---
importances = pd.Series(best_rf.feature_importances_, index=X.columns)
top20 = importances.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
top20.plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Top 20 Features — Random Forest Gini Importance',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Decrease in Gini Impurity')
plt.tight_layout()
plt.savefig('outputs/figures/rf_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# --- Logistic Regression: coefficient polarity chart ---
coef_df = pd.DataFrame({
    'Feature':     X.columns,
    'Coefficient': best_lr.coef_[0]
}).sort_values('Coefficient', ascending=False)

coef_df['Color'] = coef_df['Coefficient'].apply(
    lambda x: 'steelblue' if x > 0 else 'indianred'
)

plt.figure(figsize=(10, 7))
plt.barh(coef_df['Feature'], coef_df['Coefficient'],
         color=coef_df['Color'])
plt.axvline(0, color='black', linewidth=1)
plt.title('Logistic Regression Feature Coefficients\n'
          'Blue = increases diabetes risk  |  Red = decreases risk',
          fontsize=12, fontweight='bold')
plt.xlabel('Coefficient Value')
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('outputs/figures/lr_coefficients.png', dpi=150)
plt.show()

## 8 — Save Models

In [ ]:
joblib.dump(best_rf,  'outputs/models/random_forest.pkl')
joblib.dump(best_lr,  'outputs/models/logistic_regression.pkl')
joblib.dump(scaler,   'outputs/models/scaler.pkl')

print('Models saved:')
print('  outputs/models/random_forest.pkl')
print('  outputs/models/logistic_regression.pkl')
print('  outputs/models/scaler.pkl')

# Reload example:
# model  = joblib.load('outputs/models/random_forest.pkl')
# scaler = joblib.load('outputs/models/scaler.pkl')
# X_new_sc = scaler.transform(X_new)
# predictions = model.predict(X_new_sc)